___
# <center>Atividade: Encadear Operações</center>
___

## Aula 04

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * escrever uma análise como uma sequência de operações encadeadas;
 * usar os quatro verbos: escolher linhas, escolher colunas, ordenar e agregar por grupo;
 * explicar por que trocar a ordem de duas operações muda o resultado;
 * criar uma coluna nova antes de encadear.

Na **Gincana do Pipeline** você montou estes encadeamentos com tiras de papel,
sem digitar uma linha. Este notebook é o mesmo conteúdo escrito em python, um
verbo de cada vez, e serve para estudar por conta: em sala nós rodamos só as
rodadas da gincana, no notebook da aula 5.

A tradução das peças é direta:

| a carta no tabuleiro | no código |
|---|---|
| separar as linhas que passam | `.query("...")` |
| ler só algumas colunas | `[["a", "b"]]` |
| enfileirar em ordem | `.sort_values("a")` |
| fazer as pilhas e resumir cada uma | `.groupby("a").agg(...)` |
| criar uma coluna nova | `.assign(nova=...)` |

> 📌 Este notebook é a referência do **Projeto 02**, na quinta-feira. Se você só
> for rodar uma coisa em casa, rode a seção *A ordem importa*.


___
<div id="indice"></div>

## Índice

- [Reincidência e regime inicial](#problema)

- [Por que encadear](#problema-encadeamento)
    - [🔗 A mesma coisa, encadeada](#encadeada)
    - [( ) Por que os parênteses](#parenteses)

- [Os quatro verbos](#verbos)
    - [1️⃣ .query(): escolher linhas](#query)
    - [2️⃣ [[...]]: escolher colunas](#colunas)
    - [3️⃣ .sort_values(): ordenar](#sort)
    - [4️⃣ .groupby() e .agg(): agregar por grupo](#groupby)

- [Mais duas operações de apoio](#apoio)

- [A ordem importa](#ordem)

- [Criar uma coluna nova](#coluna)

- [Montando o pipeline](#pipeline)
    - [EXERCÍCIO 1: resumo por câmara](#ex1)

- [Exercícios](#exercicios)
    - [EXERCÍCIO 2: pena por regime](#ex2)
    - [EXERCÍCIO 3: as cinco maiores comarcas](#ex3)

- [RESUMO](#resumo)


___
<div id="problema"></div>

# Reincidência e regime inicial

A pergunta de hoje é a mesma da mesa, agora na base inteira:

> Nas apelações criminais do TJSP, a proporção de acórdãos que mencionam
> reincidência varia conforme o regime inicial fixado?

**As variáveis da base:**

* `processo`, `cd_acordao`: identificadores.
* `classe`, `assunto`, `relator`, `comarca`, `orgao_julgador`, `camara`: dados do julgamento.
* `data_julgamento`, `data_publicacao`: datas.
* `regime_inicial`: aberto, semiaberto ou fechado, lido da ementa.
* `pena_anos`: pena em anos, lida da ementa.
* `houve_reincidencia`, `houve_confissao`, `eh_trafico`: indicadores lidos da ementa.
* `n_palavras_ementa`: tamanho da ementa.

Coletada com a biblioteca
[juscraper](https://github.com/jtrecenti/juscraper). **Não precisa rodar:**

```python
import juscraper as jus

tjsp = jus.scraper("tjsp")
acordaos = tjsp.cjsg('"apelacao criminal" E "regime inicial"', paginas=range(1, 26))
```


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"


In [ ]:
criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")
criminal.head(3)


In [ ]:
criminal.info()


> ⚠️ `regime_inicial` e `pena_anos` foram lidos do texto da ementa, e nenhum dos
> dois vem completo: o regime aparece em cerca de 70% dos acórdãos e a pena em
> 45%. `pena_anos` ainda traz valores implausíveis, porque a leitura pega o
> primeiro número seguido de "anos" que encontra. É a carta A10, agora com 475
> linhas em volta.


[Volta ao Índice](#indice)


___
<div id="problema-encadeamento"></div>

# Por que encadear

Do jeito que fizemos até a aula 3, com uma variável nova a cada operação, a
resposta sai assim:


In [ ]:
apelacoes = criminal[criminal["classe"] == "Apelação Criminal"]
com_regime = apelacoes.dropna(subset=["regime_inicial"])
fechado = com_regime[com_regime["regime_inicial"] == "fechado"]
semiaberto = com_regime[com_regime["regime_inicial"] == "semiaberto"]
aberto = com_regime[com_regime["regime_inicial"] == "aberto"]

pd.Series({
    "fechado": fechado["houve_reincidencia"].mean(),
    "semiaberto": semiaberto["houve_reincidencia"].mean(),
    "aberto": aberto["houve_reincidencia"].mean(),
}).round(3)


Funciona, e tem três problemas:

1. **seis variáveis** que existem só para chegar num resultado, e que continuam
   ocupando memória e atrapalhando a leitura do resto do notebook;
2. **nomes intermediários** como `com_regime`, que não querem dizer nada e que
   você vai reaproveitar por engano daqui a três células;
3. **não escala**: se aparecesse um quarto regime, seria preciso escrever mais
   uma linha e lembrar de incluí-la no resultado.


<div id="encadeada"></div>

### 🔗 A mesma coisa, encadeada


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["regime_inicial"])
    .groupby("regime_inicial")
    .agg(proporcao=("houve_reincidencia", "mean"))
    .round(3)
)


Leia de cima para baixo, como você leu o tabuleiro: pegue `criminal`, fique só
com as apelações, descarte quem não tem regime, faça as pilhas por regime, e
calcule a proporção de cada pilha. Nenhuma variável intermediária, e a ordem das
operações é a ordem das linhas.


<div id="parenteses"></div>

### ( ) Por que os parênteses

Em python, dentro de um par de parênteses você pode quebrar a linha à vontade.
Sem eles, `criminal` seguido de uma quebra de linha e `.query(...)` é erro de
sintaxe. Os parênteses existem só para deixar você pôr uma operação por linha,
e são o equivalente do tabuleiro que você usou na dinâmica.

O formato que vamos usar sempre é este:

```python
resultado = (
    tabela
    .operacao_1(...)
    .operacao_2(...)
)
```

Abre parêntese, o nome da tabela sozinho na primeira linha, e daí em diante uma
operação por linha, cada uma começando com ponto.


[Volta ao Índice](#indice)


___
<div id="verbos"></div>

# Os quatro verbos

São os quatro que você já executou com as cartas. Um de cada vez.


<div id="query"></div>

### 1️⃣ .query(): escolher linhas


Separar as cartas que passam na condição. A condição vai escrita como **texto**:
dentro das aspas, os nomes das colunas aparecem sem `df[...]`, e o valor
comparado vai entre aspas simples.

✔️ **Uso do `.query()`**

```python
# Sintaxe geral:
DataFrame.query("coluna == 'valor'")
```

Documentação oficial: [.query()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.query.html)


In [ ]:
criminal.query("regime_inicial == 'fechado'").shape


`.shape` devolve o par (linhas, colunas). É a forma rápida de conferir quantas
cartas sobraram na mesa.

Para combinar condições, use `and`, `or` e `not`, por extenso:


In [ ]:
criminal.query("regime_inicial == 'fechado' and houve_reincidencia").shape


**✍️ Agora você.** Fique só com os acórdãos de tráfico em que houve confissão.


In [ ]:
criminal.query("eh_trafico ________ houve_confissao").shape


<div id="colunas"></div>

### 2️⃣ [[...]]: escolher colunas

Ler só algumas colunas de cada carta. Duas chaves com uma lista de nomes dentro
devolvem as colunas pedidas, na ordem em que você pediu.


In [ ]:
(
    criminal
    [["processo", "comarca", "regime_inicial", "pena_anos"]]
    .head(3)
)


> ⚠️ São **dois** pares de colchetes. Um só, como em `criminal["comarca"]`,
> devolve uma coluna solta, e não uma tabela. Com uma coluna solta o
> encadeamento acaba ali.


**✍️ Agora você.** Devolva só `processo`, `camara` e `houve_reincidencia`.


In [ ]:
(
    criminal
    [["processo", "________", "houve_reincidencia"]]
    .head(3)
)


<div id="sort"></div>

### 3️⃣ .sort_values(): ordenar


Enfileirar as cartas. O primeiro argumento diz por qual coluna ordenar, e
`ascending=False` inverte para o maior primeiro.

✔️ **Uso do `.sort_values()`**

```python
# Sintaxe geral:
DataFrame.sort_values("coluna", ascending=False)
```

Documentação oficial: [.sort_values()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)


In [ ]:
(
    criminal
    .sort_values("n_palavras_ementa", ascending=False)
    [["processo", "comarca", "n_palavras_ementa"]]
    .head(5)
)


**✍️ Agora você.** Ordene pela pena, da maior para a menor, e olhe as cinco primeiras. É a carta A10 de novo: a leitura automática da pena erra em alguns acórdãos.


In [ ]:
(
    criminal
    .sort_values("________", ascending=________)
    [["processo", "pena_anos", "regime_inicial"]]
    .head(5)
)


<div id="groupby"></div>

### 4️⃣ .groupby() e .agg(): agregar por grupo

Fazer as pilhas e virar fichas. `.groupby("coluna")` separa a tabela em pedaços,
um por valor da coluna, e `.agg(...)` calcula uma estatística em cada pedaço,
devolvendo **uma linha por grupo**.


As estatísticas são as mesmas da aula 3, agora escritas como texto: `"mean"`,
`"median"`, `"std"`, `"min"`, `"max"`, `"sum"`, `"nunique"`, além de `"size"`,
que conta as linhas do grupo.

✔️ **Uso do `.groupby().agg()`**

```python
# Sintaxe geral:
DataFrame.groupby("coluna_de_grupo").agg(nome_da_saida=("coluna_de_entrada", "estatistica"))
```

Documentação oficial: [.groupby().agg()](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html)


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(
        n=("processo", "size"),
        mediana_palavras=("n_palavras_ementa", "median"),
    )
)


Repare que `regime_inicial` saiu **fora** da tabela, à esquerda, em negrito:
depois de um `groupby`, a coluna de agrupamento vira o **índice** do resultado, e
não uma coluna normal. Isso atrapalha se você quiser continuar encadeando.

O `.reset_index()` traz o índice de volta para dentro da tabela. Por isso ele
aparece no fim de quase todo `groupby`: com o índice de volta, dá para filtrar e
ordenar o resultado como qualquer outra tabela.


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(n=("processo", "size"))
    .reset_index()
)


E vale lembrar da aula 3: a média de uma coluna de verdadeiro e falso é a
proporção. Foi assim que você preencheu a ficha-resumo, contando quantas cartas
tinham "sim" e dividindo pelo tamanho da pilha.


**✍️ Agora você.** Acrescente ao resumo a proporção de acórdãos com reincidência e a proporção de tráfico.


In [ ]:
(
    criminal
    .groupby("regime_inicial")
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "________"),
        prop_trafico=("________", "mean"),
    )
    .reset_index()
    .round(3)
)


[Volta ao Índice](#indice)


___
<div id="apoio"></div>

# Mais duas operações de apoio

Não são verbos novos, são utilidades que aparecem o tempo todo:

* `.dropna(subset=["coluna"])` descarta as linhas em que aquela coluna está
  vazia. Serve para tirar da mesa os acórdãos em que a leitura da ementa não
  achou o regime ou a pena.
* `.head(n)` fica com as `n` primeiras linhas **da tabela como ela está naquele
  ponto**. Guarde esta frase, porque a próxima seção é sobre ela.


In [ ]:
(
    criminal
    .dropna(subset=["pena_anos"])
    .shape
)


[Volta ao Índice](#indice)


___
<div id="ordem"></div>

# A ordem importa

Esta é a parte que você já descobriu com as cartas. As duas células abaixo têm
exatamente as mesmas operações, e só trocam duas linhas de lugar.

Primeiro do jeito certo: ordenar e **depois** cortar.


In [ ]:
(
    criminal
    .dropna(subset=["pena_anos"])
    .query("not eh_trafico")
    .sort_values("pena_anos", ascending=False)
    .head(3)
    [["processo", "comarca", "pena_anos"]]
)


Agora cortando **antes** de ordenar:


In [ ]:
(
    criminal
    .dropna(subset=["pena_anos"])
    .query("not eh_trafico")
    .head(3)
    .sort_values("pena_anos", ascending=False)
    [["processo", "comarca", "pena_anos"]]
)


O segundo resultado não é "as três maiores penas". É "as três primeiras linhas
da base, ordenadas entre si", que é uma pergunta que ninguém fez.

O motivo é o mesmo do tabuleiro: cada operação enxerga a tabela que a operação
anterior deixou. `.head(3)` não sabe nada sobre ordenar, ele só corta o que está
na sua frente.

> 🤔 Repare também no que apareceu no primeiro resultado: uma pena de 75,3 anos.
> O código está certo e o dado está errado. Olhar os extremos antes de acreditar
> no resultado é parte do trabalho.


[Volta ao Índice](#indice)


___
<div id="coluna"></div>

# Criar uma coluna nova

Na carta havia um espaço em branco embaixo, para escrever uma coluna nova. Aqui
é a mesma coisa, e por enquanto vamos fazer isso **antes** do encadeamento, numa
linha só:


In [ ]:
criminal["pena_meses"] = criminal["pena_anos"] * 12

criminal[["processo", "pena_anos", "pena_meses"]].head(3)


A conta vale para a tabela inteira, linha por linha, sem `for` nenhum. É a mesma
ideia de escrever a coluna nova em todas as 24 cartas de uma vez.

Isso também serve para declarar a ordem de uma categórica, como na aula 2. O
regime é **ordinal**, e queremos a tabela na ordem aberto, semiaberto, fechado, e
não em ordem alfabética:


In [ ]:
criminal["regime"] = pd.Categorical(
    criminal["regime_inicial"],
    categories=["aberto", "semiaberto", "fechado"],
    ordered=True,
)

criminal["regime"].dtype


**✍️ Agora você.** Crie a coluna `ementa_longa`, verdadeira quando a ementa tiver mais de 200 palavras.


In [ ]:
criminal["ementa_longa"] = criminal["________"] > 200

criminal[["processo", "n_palavras_ementa", "ementa_longa"]].head(3)


> 🤔 Existe um jeito de criar a coluna **dentro** do encadeamento, com
> `.assign()`. Ele é útil quando a coluna nova depende de um filtro que veio
> antes, e vamos deixar para quando essa necessidade aparecer. Por enquanto,
> coluna nova é uma linha antes do parêntese.


[Volta ao Índice](#indice)


___
<div id="pipeline"></div>

# Montando o pipeline

Com a coluna `regime` já criada, a resposta da pergunta de hoje cabe em seis
linhas. O `observed=True` no `groupby` existe porque a coluna é categórica: sem
ele, o pandas devolveria também as categorias que não aparecem em nenhuma linha.


In [ ]:
resumo = (
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["regime"])
    .groupby("regime", observed=True)
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "mean"),
        prop_confissao=("houve_confissao", "mean"),
        prop_trafico=("eh_trafico", "mean"),
    )
    .reset_index()
    .round(3)
)

resumo


A leitura é direta, e é a mesma conclusão a que a mesa chegou: a menção a
reincidência sobe conforme o regime fica mais severo. Não é surpresa, é quase a
definição legal do regime, e serve para conferir que a leitura das variáveis está
coerente.

> 🤔 E o que **não** dá para concluir: nada sobre causalidade, e nada sobre os
> acórdãos em que o regime não foi identificado, que são cerca de 30% da base.

Guarde a tabela `resumo`. Ela volta na aula 5, virando gráfico.


<div id="ex1"></div>

### EXERCÍCIO 1

Monte um resumo parecido, agora por `camara`, mantendo só as câmaras com pelo
menos 15 acórdãos e ordenando da maior proporção de reincidência para a menor.
Você vai precisar de `.reset_index()`, `.query()` e `.sort_values()` **depois**
do `.agg()`.

> 🤔 Na aula 3 vimos que `camara` é a metade de um código, e que sozinha não
> nomeia um órgão. Aqui ela serve: esta base só tem câmaras criminais, então não
> existem duas 6ª câmaras para confundir. O tipo de uma variável depende da
> tabela em que ela está, e não só do nome dela.


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .dropna(subset=["camara"])
    .groupby("________")
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "________"),
    )
    .________()
    .query("n >= ________")
    .sort_values("________", ascending=False)
    .round(3)
)


[Volta ao Índice](#indice)


___
<div id="exercicios"></div>

# Exercícios


<div id="ex2"></div>

### EXERCÍCIO 2

A pena lida da ementa tem valores implausíveis, como penas acima de 40 anos, que
vêm de a leitura pegar um número errado. Monte um encadeamento que descarte as
penas ausentes e as maiores que 30 anos, e devolva mediana, média e desvio padrão
da pena por regime inicial.


In [ ]:
(
    criminal
    .dropna(subset=["pena_anos", "regime"])
    .query("pena_anos ________ 30")
    .groupby("________", observed=True)
    .agg(
        n=("processo", "size"),
        mediana=("pena_anos", "________"),
        media=("pena_anos", "mean"),
        desvio=("pena_anos", "________"),
    )
    .reset_index()
    .round(2)
)


Compare a mediana com a média em cada regime. Em todos eles a média fica acima da
mediana, e a diferença é maior justamente onde as penas são mais curtas. Isso é
assimetria à direita: uns poucos valores altos puxam a média e não mexem na
mediana, que foi exatamente o que a carta A10 fez na sua pilha.


<div id="ex3"></div>

### EXERCÍCIO 3

Quais são as cinco comarcas com mais apelações criminais nesta base, e qual a
proporção de tráfico em cada uma?


In [ ]:
(
    criminal
    .query("classe == 'Apelação Criminal'")
    .groupby("________")
    .agg(n=("processo", "size"), prop_trafico=("eh_trafico", "________"))
    .reset_index()
    .sort_values("________", ascending=False)
    .head(________)
    .round(3)
)


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

Uma análise descritiva é uma sequência de operações, escrita de cima para baixo
dentro de um par de parênteses, com uma operação por linha.

| verbo | na mesa | para quê |
|---|---|---|
| `.query("...")` | separar cartas | escolher linhas por uma condição |
| `[["a", "b"]]` | ler só algumas colunas | escolher colunas |
| `.sort_values("a", ascending=False)` | enfileirar | ordenar |
| `.groupby("a").agg(saida=("b", "mean"))` | pilhas viram fichas | uma linha por grupo |
| `.dropna(subset=[...])` | tirar cartas incompletas | descartar linhas sem valor |
| `.head(n)` | pegar as n primeiras da fila | cortar |
| `.reset_index()` | | tirar o agrupamento do índice |


In [ ]:
#=> COLUNA NOVA: uma linha antes do parêntese, vale para a tabela inteira
criminal["regime"] = pd.Categorical(
    criminal["regime_inicial"],
    categories=["aberto", "semiaberto", "fechado"],
    ordered=True,
)

#=> O FORMATO: abre parêntese, tabela sozinha, uma operação por linha
resumo = (
    criminal

    #=> ESCOLHER LINHAS: condição como texto, colunas sem aspas
    .query("classe == 'Apelação Criminal'")

    #=> DESCARTAR FALTANTES de uma coluna
    .dropna(subset=["regime"])

    #=> AGRUPAR: observed=True descarta categorias sem nenhuma linha
    .groupby("regime", observed=True)

    #=> AGREGAR: saida=("coluna_de_entrada", "estatistica")
    .agg(
        n=("processo", "size"),
        prop_reincidencia=("houve_reincidencia", "mean"),
    )

    #=> TIRAR O AGRUPAMENTO DO ÍNDICE, para poder continuar encadeando
    .reset_index()

    #=> ORDENAR e ARREDONDAR
    .sort_values("prop_reincidencia", ascending=False)
    .round(3)
)

resumo


**Duas regras que valem sempre:**

1. Cada operação enxerga a tabela que a anterior deixou. Antes de escrever a
   próxima linha, pergunte o que está na mesa naquele ponto.
2. Quando a sequência passar de umas oito linhas, quebre em duas partes, com um
   nome que signifique alguma coisa, como fizemos com `resumo`.


[Volta ao Índice](#indice)
